In [1]:
from __future__ import annotations

import sys
from pathlib import Path


import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

In [3]:
from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.evaluation.metrics import evaluate
from ebm_unlearning.src.evaluation.classification import evaluate_classification
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.utils.seed import set_seed

In [4]:
with open(ROOT / "configs" / "config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))

device = torch.device(cfg.get("device", "cpu"))

In [5]:
# Data splits
spec = DatasetSpec(name=cfg["data"]["dataset"], data_dir=str(ROOT / cfg["data"]["data_dir"]), train=True, download=True)
dset = load_dataset(spec)

forget_spec = ForgetSpec(mode=cfg["data"]["forget"]["mode"], class_label=cfg["data"]["forget"]["class_label"])
retain_spec = RetainSpec(mode=cfg["data"]["retain"]["mode"])
forget_all, retain_all = split_forget_retain(dset, forget_spec, retain_spec)

holdout_fraction = float(cfg["evaluation"]["holdout_fraction"])
forget_train, forget_holdout = train_holdout_split(forget_all, holdout_fraction, seed=int(cfg["seed"]))
retain_train, retain_holdout = train_holdout_split(retain_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

batch_size = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])

forget_train_loader = DataLoader(forget_train, batch_size=batch_size, shuffle=False, num_workers=num_workers)
forget_holdout_loader = DataLoader(forget_holdout, batch_size=batch_size, shuffle=False, num_workers=num_workers)
retain_train_loader = DataLoader(retain_train, batch_size=batch_size, shuffle=False, num_workers=num_workers)
retain_holdout_loader = DataLoader(retain_holdout, batch_size=batch_size, shuffle=False, num_workers=num_workers)

[data] loading cifar10 (train=True, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
Files already downloaded and verified


In [ ]:
# Models
E0 = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
)
E0 = load_pretrained(E0, str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)

E = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
)
E = load_pretrained(E, str(ROOT / cfg["unlearning"]["checkpoint_path"]), device=device)

metrics = evaluate(
    E,
    E0,
    forget_train_loader=forget_train_loader,
    forget_holdout_loader=forget_holdout_loader,
    retain_train_loader=retain_train_loader,
    retain_holdout_loader=retain_holdout_loader,
    device=device,
)

print("=== Energy-based unlearning metrics (holdout from train split) ===")
for k in sorted(metrics):
    print(f"{k}: {metrics[k]}")

# Classification-style evaluation on TEST split via argmin_y E(x,y)
print("\n=== Classification via argmin_y E(x,y) on TEST split ===")

test_spec = DatasetSpec(
    name=cfg["data"]["dataset"],
    data_dir=str(ROOT / cfg["data"]["data_dir"]),
    train=False,
    download=True,
)
test_dset = load_dataset(test_spec)
test_loader = DataLoader(test_dset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

num_classes = int(cfg["model"].get("num_classes", 10))
forget_label = int(cfg["data"]["forget"]["class_label"])

cls_pre = evaluate_classification(E0, test_loader, device=device, num_classes=num_classes, forget_label=forget_label, y_chunk=10)
cls_unl = evaluate_classification(E, test_loader, device=device, num_classes=num_classes, forget_label=forget_label, y_chunk=10)

print("[pretrained] overall acc:", cls_pre.overall_accuracy)
print("[pretrained] forget acc:", cls_pre.forget_accuracy, "retain acc:", cls_pre.retain_accuracy)
print("[unlearned ] overall acc:", cls_unl.overall_accuracy)
print("[unlearned ] forget acc:", cls_unl.forget_accuracy, "retain acc:", cls_unl.retain_accuracy)

print("\nPer-class accuracy (pretrained):")
for c in range(num_classes):
    print(c, ":", cls_pre.per_class_accuracy[c])

print("\nPer-class accuracy (unlearned):")
for c in range(num_classes):
    print(c, ":", cls_unl.per_class_accuracy[c])

print("\nConfusion matrix (pretrained):")
print(cls_pre.confusion)

print("\nConfusion matrix (unlearned):")
print(cls_unl.confusion)



/home/owais/machine unlearning/ebm_unlearning/src/training/pretrain.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location=devi

=== Classification via argmin_y E(x,y) on TEST split ===
[data] loading cifar10 (train=False, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
Files already downloaded and verified
[pretrained] overall acc: 0.6942
[pretrained] forget acc: 0.787 retain acc: 0.6838888888888889

Per-class accuracy (pretrained):
0 : 0.787
1 : 0.742
2 : 0.579
3 : 0.573
4 : 0.679
5 : 0.541
6 : 0.792
7 : 0.761
8 : 0.734
9 : 0.754

Confusion matrix (pretrained):
[[787  15  35  14  23   3   5  16  68  34]
 [ 35 742  13  18   1   8  12   7  37 127]
 [ 77   4 579  72 108  42  77  31   8   2]
 [ 36  10  62 573  61 114  80  42   8  14]
 [ 34   4  47  61 679  43  57  68   6   1]
 [ 16   6  37 233  44 541  33  73   3  14]
 [ 12   7  40  53  55  18 792  12   3   8]
 [ 24   7  21  49  54  46  13 761   4  21]
 [109  50  19  19  16   3   7   8 734  35]
 [ 44  90  11  29   4  15   5  23  25 754]]

[info] unlearned checkpoint not found at: /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoi